# Parallel Ensemble Inference Engine — Offline Training
**HPC Project · Andrea · University of Messina**

This notebook trains three classifiers (KNN, Naive Bayes, Logistic Regression) on a synthetic
binary classification dataset and exports all model parameters as plain-text files.
The C++ inference engine loads these files at startup and performs inference only — no training happens at runtime.


In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import os

In [2]:
# ── Dataset configuration ─────────────────────────────────────────────────────
# I want to generate a synthetic binary classification dataset with:
#   - 70 000 total samples  (50k train / 20k test)
#   - 20 features, 10 of which are truly informative
#   - 2 redundant features (linear combinations of informative ones)
#   - class_sep=1.0 keeps the problem non-trivial but solvable by all three models
#   - random_state=42 ensures full reproducibility across runs
#
# The test set is NOT exported: the C++ engine regenerates it on-the-fly
# using the same seed, scaling freely from 10k to 1M samples.

RANDOM_STATE = 42
N_SAMPLES    = 70_000
N_FEATURES   = 20
N_INFORMATIVE = 10
N_REDUNDANT   = 2
CLASS_SEP     = 1.0
TEST_SIZE     = 20_000   # held-out for accuracy validation only
KNN_K         = 5        # number of neighbours — exported so C++ uses the same k

X, y = make_classification(
    n_samples     = N_SAMPLES,
    n_features    = N_FEATURES,
    n_informative = N_INFORMATIVE,
    n_redundant   = N_REDUNDANT,
    n_classes     = 2,
    class_sep     = CLASS_SEP,
    random_state  = RANDOM_STATE
)

print(f"Dataset shape : {X.shape}")
print(f"Class balance : {np.bincount(y)}")

Dataset shape : (70000, 20)
Class balance : [35010 34990]


In [3]:
# ── Train / test split ────────────────────────────────────────────────────────
# 50 000 samples go to training (used by KNN at inference time and to fit NB/LR).
# 20 000 samples are held out for accuracy validation in this notebook only.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y       # preserve class balance in both splits
)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

Training set : (50000, 20)
Test set     : (20000, 20)


In [4]:
# ── Feature scaling ───────────────────────────────────────────────────────────
# StandardScaler (zero mean, unit variance) is required for both KNN (distance-based)
# and Logistic Regression (gradient-sensitive). Naive Bayes is scale-invariant but
# we apply the same scaler for consistency — the exported means/variances already
# reflect the scaled space, so the C++ engine must apply the same transform.
#
# IMPORTANT: fit on training data only, then transform both splits.

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Save scaler parameters so the C++ engine can replicate the transform
# on the synthetically generated test samples at runtime.
scaler_mean = scaler.mean_          # shape (n_features,)
scaler_std  = scaler.scale_         # shape (n_features,)

print("Scaler fitted. Mean[:3]:", scaler_mean[:3].round(4))


Scaler fitted. Mean[:3]: [-0.0011 -0.004  -0.5043]


In [5]:
# ── Model training ────────────────────────────────────────────────────────────
# All three models are fitted on the scaled training set.
# KNN is a lazy learner — 'training' just stores X_train.
# NB and LR learn compact parameter sets that are exported as small text files.

# K-Nearest Neighbours
knn = KNeighborsClassifier(n_neighbors=KNN_K, metric='euclidean', algorithm='brute')
knn.fit(X_train, y_train)

# Gaussian Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)

# Logistic Regression (liblinear solver, well-suited for binary classification)
lr = LogisticRegression(max_iter=1000, solver='liblinear', random_state=RANDOM_STATE)
lr.fit(X_train, y_train)

print("All models trained.")

All models trained.


In [6]:
# ── Accuracy validation ───────────────────────────────────────────────────────
# Evaluate each model individually and as a soft-voting ensemble.
# This validates that the exported parameters will produce correct predictions
# in C++ — if individual accuracies look wrong here, something is off before export.

# Individual predictions
y_pred_knn = knn.predict(X_test)
y_pred_nb  = nb.predict(X_test)
y_pred_lr  = lr.predict(X_test)

# Soft-voting ensemble: average predicted probabilities, threshold at 0.5
p_knn = knn.predict_proba(X_test)[:, 1]
p_nb  = nb.predict_proba(X_test)[:, 1]
p_lr  = lr.predict_proba(X_test)[:, 1]
y_pred_ensemble = ((p_knn + p_nb + p_lr) / 3 >= 0.5).astype(int)

print(f"KNN accuracy      : {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Naive Bayes acc.  : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Logistic Reg. acc.: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Ensemble accuracy : {accuracy_score(y_test, y_pred_ensemble):.4f}")
print()
print("Ensemble classification report:")
print(classification_report(y_test, y_pred_ensemble))

KNN accuracy      : 0.9325
Naive Bayes acc.  : 0.7964
Logistic Reg. acc.: 0.8050
Ensemble accuracy : 0.8844

Ensemble classification report:
              precision    recall  f1-score   support

           0       0.90      0.87      0.88     10003
           1       0.87      0.90      0.89      9997

    accuracy                           0.88     20000
   macro avg       0.88      0.88      0.88     20000
weighted avg       0.88      0.88      0.88     20000



In [7]:
# ── Export directory ──────────────────────────────────────────────────────────
# All files land in ./model_params/. Download this folder and place it
# next to the C++ binary (or set MODEL_DIR in the Makefile).

OUT_DIR = "model_params"
os.makedirs(OUT_DIR, exist_ok=True)

In [8]:
# ── Export: dataset metadata ──────────────────────────────────────────────────
# A single config file read by C++ at startup to know array dimensions,
# the value of k, and the random seed used to regenerate the test set.

with open(f"{OUT_DIR}/config.txt", "w") as f:
    f.write(f"n_features {N_FEATURES}\n")
    f.write(f"n_train {X_train.shape[0]}\n")
    f.write(f"knn_k {KNN_K}\n")
    f.write(f"random_state {RANDOM_STATE}\n")
    f.write(f"n_classes 2\n")

print("config.txt written.")

config.txt written.


In [9]:
# ── Export: StandardScaler parameters ────────────────────────────────────────
# The C++ engine must apply the same z-score transform to inference samples
# before computing distances or dot products.
# Format: one value per line, matching feature order.

np.savetxt(f"{OUT_DIR}/scaler_mean.txt", scaler_mean)
np.savetxt(f"{OUT_DIR}/scaler_std.txt",  scaler_std)

print(f"scaler_mean.txt : {scaler_mean.shape}")
print(f"scaler_std.txt  : {scaler_std.shape}")

scaler_mean.txt : (20,)
scaler_std.txt  : (20,)


In [10]:
# ── Export: KNN training data ─────────────────────────────────────────────────
# KNN is a lazy learner: the full training set IS the model.
# X_train.txt : 50 000 rows × 20 features, space-separated floats
# y_train.txt : 50 000 integer labels (0 or 1), one per line
#
# These are the largest files (~8 MB total). The C++ engine loads them once
# at startup into a contiguous array for cache-friendly distance computation.

np.savetxt(f"{OUT_DIR}/X_train.txt", X_train, fmt="%.8f")
np.savetxt(f"{OUT_DIR}/y_train.txt", y_train, fmt="%d")

print(f"X_train.txt : {X_train.shape}  →  {os.path.getsize(OUT_DIR+'/X_train.txt')/1e6:.1f} MB")
print(f"y_train.txt : {y_train.shape}")

X_train.txt : (50000, 20)  →  11.5 MB
y_train.txt : (50000,)


In [11]:
# ── Export: Gaussian Naive Bayes parameters ───────────────────────────────────
np.savetxt(f"{OUT_DIR}/nb_means.txt",  nb.theta_,        fmt="%.8f")
np.savetxt(f"{OUT_DIR}/nb_vars.txt",   nb.var_,          fmt="%.8f")
np.savetxt(f"{OUT_DIR}/nb_priors.txt", nb.class_prior_,  fmt="%.8f")

print(f"nb_means.txt  : {nb.theta_.shape}")
print(f"nb_vars.txt   : {nb.var_.shape}")
print(f"nb_priors.txt : {nb.class_prior_.shape}  (linear priors)")

nb_means.txt  : (2, 20)
nb_vars.txt   : (2, 20)
nb_priors.txt : (2,)  (linear priors)


In [12]:
# ── Export: Logistic Regression parameters ────────────────────────────────────
# For binary classification, scikit-learn stores a single weight vector
# of shape (1, n_features) and a scalar bias.
#
# C++ inference: probability of class 1 = sigmoid(w · x + b)
# where sigmoid(z) = 1 / (1 + exp(-z)).
# Threshold at 0.5 → predict class 1 if w · x + b > 0.

np.savetxt(f"{OUT_DIR}/lr_weights.txt", lr.coef_,      fmt="%.8f")
np.savetxt(f"{OUT_DIR}/lr_bias.txt",   lr.intercept_, fmt="%.8f")

print(f"lr_weights.txt : {lr.coef_.shape}")
print(f"lr_bias.txt    : {lr.intercept_.shape}")

lr_weights.txt : (1, 20)
lr_bias.txt    : (1,)


In [13]:
# ── Export: test set as flat binary file ──────────────────────────────────────
# Format: [int32 n_samples][int32 n_features][float64 × n_samples × n_features]
# C++ loads this directly into a contiguous array — cache-friendly and exact.
# y_test exported separately as int32 array for accuracy validation.

import struct

def export_binary(path, arr):
    with open(path, 'wb') as f:
        rows, cols = arr.shape
        f.write(struct.pack('ii', rows, cols))
        f.write(arr.astype(np.float64).tobytes())

def export_labels(path, arr):
    with open(path, 'wb') as f:
        f.write(struct.pack('i', len(arr)))
        f.write(arr.astype(np.int32).tobytes())

export_binary(f"{OUT_DIR}/X_test.bin", X_test)
export_labels(f"{OUT_DIR}/y_test.bin", y_test)

print(f"X_test.bin : {X_test.shape}  →  {os.path.getsize(OUT_DIR+'/X_test.bin')/1e6:.1f} MB")
print(f"y_test.bin : {y_test.shape}")

X_test.bin : (20000, 20)  →  3.2 MB
y_test.bin : (20000,)


In [14]:
# ── Summary ───────────────────────────────────────────────────────────────────
# List all exported files with sizes for a final sanity check.

print(f"{'File':<25} {'Size':>10}")
print("-" * 37)
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    size  = os.path.getsize(fpath)
    unit  = "KB" if size < 1_000_000 else "MB"
    val   = size / 1_000 if size < 1_000_000 else size / 1_000_000
    print(f"{fname:<25} {val:>8.1f} {unit}")

print()
print("All parameters exported. Download the model_params/ folder and place")
print("it in the same directory as the C++ binary.")

File                            Size
-------------------------------------
X_test.bin                     3.2 MB
X_train.txt                   11.5 MB
config.txt                     0.1 KB
lr_bias.txt                    0.0 KB
lr_weights.txt                 0.2 KB
nb_means.txt                   0.5 KB
nb_priors.txt                  0.0 KB
nb_vars.txt                    0.4 KB
scaler_mean.txt                0.5 KB
scaler_std.txt                 0.5 KB
y_test.bin                    80.0 KB
y_train.txt                  100.0 KB

All parameters exported. Download the model_params/ folder and place
it in the same directory as the C++ binary.


In [15]:
import shutil
shutil.make_archive("model_params", "zip", "model_params")
print("model_params.zip ready — download it from the file browser on the left")

model_params.zip ready — download it from the file browser on the left
